# 3. Проверенные источники и переход к обучающему архиву

Ноутбук восстанавливает разведку уже проверенных источников по сохранённым каталогам и манифестам. Сетевые запросы и повторная загрузка здесь не выполняются. Сначала были собраны небольшие выборки из открытых наборов, Internet Archive и других доступных источников, затем пользователь предоставил архив шести видов спорта.

Массовый сбор через видеопоиск и разметку gpt-6-luna выполнил пользователь. Код этого внешнего сборщика в исходном проекте отсутствует; здесь сохранены переданные метаданные и описание процесса, без выдуманного скрипта сбора.

In [1]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown, Image

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "scripts/train_vk_bundle.py").is_file()
             and (p / "configs/experiment_index.csv").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Запустите ноутбук внутри sports_video_repo")

def load_json(path):
    return json.loads((ROOT / path).read_text(encoding="utf-8"))

def load_jsonl(path):
    return [json.loads(line) for line in (ROOT / path).read_text(encoding="utf-8").splitlines() if line]

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True})
print("Читаем сохранённые результаты; обучение и скачивание не запускаются.")

Читаем сохранённые результаты; обучение и скачивание не запускаются.


In [2]:
files = ["multisports_sample.csv", "muvy_sample.csv", "muvs_sample.csv",
         "internet_archive_sample.csv", "internet_archive_college_sample.csv",
         "internet_archive_middle_school_sample.csv", "wikimedia_u17_sample.csv", "sportsshot_sample.csv"]
display(pd.DataFrame([{ "манифест": name, "строк": len(pd.read_csv(ROOT / "data/metadata" / name))} for name in files]))
accepted = pd.read_csv(ROOT / "data/metadata/collection_sport_eval.csv")
print("Итоговая ранняя выборка:", len(accepted), "коротких клипа")
display(accepted.head())

,манифест,строк
0,multisports_sample.csv,30
1,muvy_sample.csv,15
2,muvs_sample.csv,23
3,internet_archive_sample.csv,31
4,internet_archive_college_sample.csv,17
5,internet_archive_middle_school_sample.csv,4
6,wikimedia_u17_sample.csv,3
7,sportsshot_sample.csv,3


Итоговая ранняя выборка: 53 коротких клипа


,clip_id,source_video_id,video_path,sport_label,source_dataset,gender_category,age_category
0,IA:NMS_vs._Batavia_8th_Grade_Girls_Basketball_...,IA:NMS_vs._Batavia_8th_Grade_Girls_Basketball_...,data/processed/internet_archive_sample/basketb...,basketball,InternetArchive,NaN,NaN
1,IA:btvnj-2_November_Men_s_Basketball_Northern_...,IA:btvnj-2_November_Men_s_Basketball_Northern_...,data/processed/internet_archive_college_sample...,basketball,InternetArchive,NaN,NaN
2,IA:cormi-7th_8th_Grade_Boys_Basketball_-_Almon...,IA:cormi-7th_8th_Grade_Boys_Basketball_-_Almon...,data/processed/internet_archive_sample/basketb...,basketball,InternetArchive,NaN,NaN
3,IA:hcectvky-8th_Grade_Boys_Basketball_Champion...,IA:hcectvky-8th_Grade_Boys_Basketball_Champion...,data/processed/internet_archive_sample/basketb...,basketball,InternetArchive,NaN,NaN
4,IA:mapsdwi-11-10-2016_8th_Grade_Girls_Basketba...,IA:mapsdwi-11-10-2016_8th_Grade_Girls_Basketba...,data/processed/internet_archive_sample/basketb...,basketball,InternetArchive,NaN,NaN


## Оценка моделей на 53 ранее собранных клипах

In [3]:
report = load_json("outputs/external_collection_53_mixed/report.json")
display(pd.DataFrame([{ "модель": name, "accuracy": item["sport"]["accuracy"],
     "Macro-F1": item["sport"]["macro_f1_observed_sports"], "секунд": item["runtime"]["elapsed_sec"]}
    for name, item in report["models"].items()]).round(3))

,модель,accuracy,Macro-F1,секунд
0,dinov2_small,0.736,0.759,13.546
1,x3d_xs,0.585,0.593,11.658
2,r3d_18,0.547,0.515,34.651


## Полученный обучающий архив

Для шести видов спорта gpt-6-luna предложила 112 общих и уточнённых поисковых запросов. Поиск дал 1884 попадания и 1310 уникальных видео. Модель получала только заголовок, описание и длительность, без поискового запроса и кадров. Она предварительно приняла 523 видео, отправила 596 в review и отклонила 191. Это слабые текстовые метки.

Из принятых записей извлекались до трёх пятисекундных окон; из окон — JPEG примерно с частотой 5 кадров/с. Источники с менее чем двумя пригодными окнами исключались. Полные видео в архиве не хранятся. В отдельных окнах могут быть паузы или заставки.

In [4]:
stats = load_json("sports_training_bundle/statistics.json")
rows = sum((load_jsonl(f"sports_training_bundle/{part}.jsonl") for part in ("train", "validation", "test")), [])
print("Видео:", len({r["video_id"] for r in rows}), "Групп:", len({r["source_group_id"] for r in rows}),
      "Окон:", len(rows), "Кадров:", sum(len(r["frames"]) for r in rows))
display(pd.DataFrame({"видео": stats["sources_by_split"], "окон": stats["clips_by_split"]}))

Видео: 404 Групп: 396 Окон: 1086 Кадров: 26015


,видео,окон
train,269,720
validation,67,179
test,68,187


Небольшая открытая выборка была полезна для проверки переноса и поиска ошибок, но сама по себе не покрывала обучение трёх голов. Предоставленный архив добавил шесть видов спорта и две категории соревнования. Его дальнейший аудит и протокол оценки показаны в следующем ноутбуке.

[Результаты раннего сбора](../docs/results/DATA_COLLECTION_RESULTS.md), [исторические аудиты источников](../docs/history/ADDITIONAL_SOURCES_AUDIT.md).